# 배터리 균등성 분석 — 비-점유 / "TEST" 시퀀스 (직접 학습)

직접 학습한 **비-점유** 정책으로, 더 긴 4단계 시퀀스 **GROUND→T→E→S→T** 에서
배터리 인지 정책이 드론 간 잔량을 더 균등하게 만드는지 분석한다.

- **NB**: 배터리 없음 (`comm_train.py`, obs 71)
- **B** : 배터리 (`comm_train_battery.py`, obs 85 = [기본71]+[배터리14])
- **공정 비교**: 비-점유라 배터리 obs 앞 71차원이 NB obs와 동일 → NB를 **같은 배터리 환경**에
  넣되 관측 앞 71차원만 사용. 두 정책이 동일 환경·동일 배터리 회계, 정책만 다름.

**런타임**: GPU. 학습 2개(시간 소요), 이미 ckpt 있으면 skip.

## 1. clone / drive / pip

In [ ]:
BRANCH="Saehoon"
%cd /content
!rm -rf /content/RL-2026s1-tp
!git clone https://github.com/umbrellalily/RL-2026s1-tp.git /content/RL-2026s1-tp
%cd /content/RL-2026s1-tp
!git switch $BRANCH && git pull origin $BRANCH
!git log --oneline -1
from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT="/content/drive/MyDrive/drone_results/battery_letters"
!mkdir -p {DRIVE_ROOT}/ckpts {DRIVE_ROOT}/runs {DRIVE_ROOT}/analysis {DRIVE_ROOT}/gifs
!pip install -q torchrl pettingzoo==1.24.3 gymnasium scipy matplotlib pillow

## 2. 설정 ("TEST" 시퀀스 + 길이에 맞춘 배터리)

In [ ]:
TARGET_SEQUENCE = "GROUND,T,E,S,T"   # 4단계 (T,E,S,T)
EXP_TAG = "TEST"
GRID_SIZE, N_AGENTS = 25, 14
MAX_STEPS = 800                      # 4단계라 더 길게
SHAPES = [s.strip() for s in TARGET_SEQUENCE.split(",") if s.strip()]

# 학습 하이퍼파라미터 (이전과 동일)
TOTAL_FRAMES, FRAMES_PER_BATCH, MINIBATCH = 800_000, 4096, 512
PPO_EPOCHS, LR, ENT_COEF, CLIP_EPS = 6, 2e-4, 0.02, 0.15
CKPT_EVERY = 5
COMPLETION_REWARD, ASSIGNED, COVDELTA, COVSTEP, HOVERP, SHAPING = 120.0, 0.4, 0.3, 0.01, 0.05, 0.5
EARLY_STOP, EARLY_PAT, SAVE_BEST = 1.0, 3, 0.5

# 배터리: 에피소드 길이에 맞춰 소모를 스케일(호버 누적 ~0.5) → 500스텝 설정과 동일한 여유
INIT  = 1.0
HOVER = round(0.5 / MAX_STEPS, 6)    # ≈ 0.000625
MOVE  = round(2.5 * HOVER, 6)        # ≈ 0.001563
PEN   = 0.05

NB_DIR = f"{DRIVE_ROOT}/ckpts/{EXP_TAG}_nb"     # 배터리 없음
B_DIR  = f"{DRIVE_ROOT}/ckpts/{EXP_TAG}_batt"   # 배터리
print("seq:", TARGET_SEQUENCE, "| MAX_STEPS:", MAX_STEPS, "| HOVER/MOVE:", HOVER, MOVE)

## 3. ckpt 헬퍼 + 두 정책 직접 학습 (이미 있으면 skip)

NB는 `comm_train.py`(배터리 없음, obs 71), B는 `comm_train_battery.py`(배터리, obs 85). 비-점유.

In [ ]:
import os, re, glob
def latest_ckpt(d):
    cand=[]
    for p in glob.glob(os.path.join(d,"ckpt_*.pt")):
        m=re.search(r"ckpt_(\d+)\.pt$",p)
        if m: cand.append((int(m.group(1)),p))
    if not cand: return None,0
    cand.sort(); return cand[-1][1],cand[-1][0]
def eval_ckpt(d):
    b=os.path.join(d,"ckpt_best.pt")
    return b if os.path.exists(b) else latest_ckpt(d)[0]

def run_if_needed(save_dir, cmd):
    if latest_ckpt(save_dir)[0]:
        print(f"[skip] 이미 학습됨: {latest_ckpt(save_dir)[0]}"); return
    os.makedirs(save_dir, exist_ok=True); print(cmd); get_ipython().system(cmd)

common = (f"--grid-size {GRID_SIZE} --n-agents {N_AGENTS} --max-steps {MAX_STEPS} "
          f"--shapes '{TARGET_SEQUENCE}' --completion-reward {COMPLETION_REWARD} "
          f"--assigned-target-reward {ASSIGNED} --coverage-delta-reward {COVDELTA} "
          f"--coverage-step-reward {COVSTEP} --hover-penalty {HOVERP} --shaping-coef {SHAPING} "
          f"--total-frames {TOTAL_FRAMES} --frames-per-batch {FRAMES_PER_BATCH} --minibatch-size {MINIBATCH} "
          f"--ppo-epochs {PPO_EPOCHS} --lr {LR} --ent-coef {ENT_COEF} --clip-eps {CLIP_EPS} "
          f"--ckpt-every {CKPT_EVERY} --early-stop-success {EARLY_STOP} --early-stop-patience {EARLY_PAT} "
          f"--save-best-above {SAVE_BEST}")

run_if_needed(NB_DIR, f"python comm_train.py {common} --save-dir {NB_DIR} --tb-logdir {DRIVE_ROOT}/runs/{EXP_TAG}_nb")
run_if_needed(B_DIR,  f"python comm_train_battery.py {common} --initial-battery {INIT} "
              f"--hover-battery-cost {HOVER} --move-battery-cost {MOVE} --low-battery-move-penalty {PEN} "
              f"--save-dir {B_DIR} --tb-logdir {DRIVE_ROOT}/runs/{EXP_TAG}_batt")

## 4. 분석 헬퍼 (먼저 실행)

`run_in_batt(actor, obs_in)`: **배터리 환경**에서 정책을 굴린다. NB(obs_in=71)는 관측 앞 71차원만
넣어 같은 환경·배터리 회계로 평가. 배터리는 환경이 실제로 추적.

In [ ]:
import torch, numpy as np
from torchrl.envs.utils import ExplorationType, set_exploration_type, step_mdp
import comm_eval_battery as ceb
from comm_eval_battery import _battery_color as bcolor
GROUP = ceb.GROUP
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def make_batt(seed=0):
    return ceb.make_env(seed=seed, device=device, grid_size=GRID_SIZE, n_agents=N_AGENTS,
        max_steps=MAX_STEPS, shapes=SHAPES, comm_fail_prob=0.0, completion_reward=COMPLETION_REWARD,
        wind_prob=0.0, wind_strength=1, randomize_wind=False,
        initial_battery=INIT, hover_battery_cost=HOVER, move_battery_cost=MOVE, low_battery_move_penalty=PEN)

def load_actor(ckpt, obs_dim):
    env,_ = make_batt(seed=0)
    a = ceb.build_actor(obs_dim, 5, N_AGENTS, 128, device)
    td = env.reset(); full = td.get((GROUP,"observation"))
    if full.shape[-1] != obs_dim: td.set((GROUP,"observation"), full[..., :obs_dim])
    with torch.no_grad(): a(td)
    a.load_state_dict(torch.load(ckpt, map_location=device)["actor"]); a.eval()
    return a

def run_in_batt(actor, obs_in, greedy=False, record=False, seed=0):
    env, base = make_batt(seed=seed); td = env.reset()
    expl = ExplorationType.MODE if greedy else ExplorationType.RANDOM
    frames=[]
    def snap(): return dict(positions=dict(base.agent_pos), batteries=dict(base.battery), target_cells=list(base.target_cells))
    if record: frames.append(snap())
    success=False
    for _ in range(base.max_steps):
        full=td.get((GROUP,"observation")); sl=full.shape[-1]!=obs_in
        if sl: td.set((GROUP,"observation"), full[..., :obs_in])
        with set_exploration_type(expl), torch.no_grad(): actor(td)
        if sl: td.set((GROUP,"observation"), full)
        td=env.step(td)
        if record: frames.append(snap())
        if bool(td.get(("next",GROUP,"done")).all().item()):
            success=bool(td.get(("next",GROUP,"terminated")).any().item()); break
        td=step_mdp(td)
    cov=getattr(base,"last_occupied_count",0)/max(1,len(base.target_cells))
    return dict(success=success, coverage=cov, final_batt=dict(base.battery), frames=frames)

nb_actor = load_actor(eval_ckpt(NB_DIR), 71)
b_actor  = load_actor(eval_ckpt(B_DIR), 85)
print("로드 완료 — NB:", eval_ckpt(NB_DIR), "| B:", eval_ckpt(B_DIR))

## 5. 성능 표 (NB vs 배터리 정책)

In [ ]:
N_EVAL=50
def perf(actor, obs_in):
    def rate(g):
        rs=[run_in_batt(actor,obs_in,greedy=g,seed=1000+i) for i in range(N_EVAL)]
        return (np.mean([r["success"] for r in rs]), np.mean([r["coverage"] for r in rs]),
                np.mean([np.mean(list(r["final_batt"].values())) for r in rs]))
    sg,cg,_=rate(True); ss,cs,fb=rate(False)
    return sg,ss,(cg+cs)/2,fb
out=[]; emit=lambda x:(out.append(x),print(x))
emit(f"=== 성능 비교 (n={N_EVAL}, {TARGET_SEQUENCE}) ===")
hdr="정책".ljust(16)+"succ(greedy)".rjust(13)+"succ(stoch)".rjust(13)+"coverage".rjust(11)+"final_batt".rjust(12)
emit(hdr); emit("-"*len(hdr))
for name,actor,oi,hb in [("NB(배터리없음)",nb_actor,71,False),("B(배터리)",b_actor,85,True)]:
    sg,ss,cov,fb=perf(actor,oi)
    emit(name.ljust(16)+f"{sg*100:.0f}%".rjust(13)+f"{ss*100:.0f}%".rjust(13)+f"{cov*100:.1f}%".rjust(11)
         +(f"{fb*100:.1f}%" if hb else "—").rjust(12))
open(f"{DRIVE_ROOT}/analysis/perf_{EXP_TAG}.txt","w",encoding="utf-8").write("\n".join(out)+"\n")
print("saved ->", f"{DRIVE_ROOT}/analysis/perf_{EXP_TAG}.txt")

## 6. 균등성 — 드론별 최종 배터리 분포 + box plot

두 정책을 같은 배터리 환경에서 N판씩 굴려 드론별 최종 잔량 수집. 균등할수록 std↓·최소↑·Jain↑.

In [ ]:
import matplotlib.pyplot as plt
N_FAIR=40
def collect(actor,oi):
    return np.array([np.array(list(run_in_batt(actor,oi,greedy=False,seed=2000+i)["final_batt"].values())) for i in range(N_FAIR)])
NBv,Bv=collect(nb_actor,71),collect(b_actor,85)
def jain(x):
    s1=x.sum(1); s2=(x**2).sum(1); return float(np.mean(s1**2/(x.shape[1]*np.maximum(s2,1e-9))))
def stats(v): return dict(mean=v.mean(),std=float(np.mean(v.std(1))),mn=float(np.mean(v.min(1))),
                          spread=float(np.mean(v.max(1)-v.min(1))),jain=jain(v))
sN,sB=stats(NBv),stats(Bv)
print(f"{'지표':<14}{'NB':>12}{'배터리정책':>14}")
for k,lab in [('mean','평균잔량'),('std','표준편차↓'),('mn','최소잔량↑'),('spread','폭(max-min)↓'),('jain','Jain↑')]:
    f=(lambda x:f"{x:.3f}") if k=='jain' else (lambda x:f"{x*100:.1f}%")
    print(f"{lab:<14}{f(sN[k]):>12}{f(sB[k]):>14}")
plt.figure(figsize=(6,5))
plt.boxplot([NBv.flatten()*100,Bv.flatten()*100],labels=["NB policy","battery policy"],showmeans=True)
plt.ylabel("final battery (%)"); plt.title(f"Per-drone final battery  [{TARGET_SEQUENCE}]"); plt.grid(axis="y",alpha=0.3)
png=f"{DRIVE_ROOT}/analysis/fairness_box_{EXP_TAG}.png"; plt.tight_layout(); plt.savefig(png,dpi=120); plt.show()
print("saved ->", png)

## 7. 병렬 GIF — 잔량 급감 드론 비교

NB에서 최종 잔량이 가장 낮은 드론을 골라 NB|배터리 정책 나란히. 자홍 링+★ 표식.

In [ ]:
import matplotlib.pyplot as plt, matplotlib.patches as patches
from matplotlib.animation import FuncAnimation, PillowWriter
SEED=7
nb_run=run_in_batt(nb_actor,71,greedy=False,record=True,seed=SEED)
b_run =run_in_batt(b_actor, 85,greedy=False,record=True,seed=SEED)
mark=min(nb_run["final_batt"],key=nb_run["final_batt"].get)
print("표식 드론:",mark,"| NB 최종:",f"{nb_run['final_batt'][mark]*100:.0f}%","| B 최종:",f"{b_run['final_batt'][mark]*100:.0f}%")
def save_parallel(hA,hB,tA,tB,mark,path,fps=4):
    n=max(len(hA),len(hB)); fig,ax=plt.subplots(1,2,figsize=(13,7),facecolor="#0a0a14")
    def one(a,h,step,title):
        f=h[min(step,len(h)-1)]; a.clear(); a.set_facecolor("#0a0a14")
        a.set_xlim(-0.5,GRID_SIZE-0.5); a.set_ylim(-0.5,GRID_SIZE-0.5); a.invert_yaxis(); a.set_aspect("equal"); a.set_xticks([]); a.set_yticks([])
        ts=set(f["target_cells"]); mb=f["batteries"].get(mark,0.0)
        a.set_title(f"{title}\nstep {min(step,len(h)-1)} | {mark} batt={mb*100:.0f}%",color="#ddd",fontsize=11)
        for (r,c) in f["target_cells"]:
            a.add_patch(patches.Rectangle((c-0.5,r-0.5),1,1,facecolor="#1c1c2e",edgecolor="#3a3a55",lw=0.6,ls=(0,(2,2))))
        for ag,(r,c) in f["positions"].items():
            on=(r,c) in ts
            a.add_patch(patches.Circle((c,r),0.34 if on else 0.22,facecolor="#ffd24a" if on else "#3a3a44",edgecolor="#fff4b3" if on else "#555",lw=1.2))
            b=f["batteries"].get(ag,0.0)
            a.add_patch(patches.Rectangle((c-0.45,r-0.78),0.9,0.13,facecolor="#222230",edgecolor="#777",lw=0.3))
            if b>0: a.add_patch(patches.Rectangle((c-0.45,r-0.78),0.9*max(0,min(1,b)),0.13,facecolor=bcolor(b),edgecolor="none"))
            if ag==mark:
                a.add_patch(patches.Circle((c,r),0.62,facecolor="none",edgecolor="#ff45ff",lw=2.6))
                a.text(c,r+0.85,"★",color="#ff45ff",ha="center",va="top",fontsize=12,fontweight="bold")
    def draw(s): one(ax[0],hA,s,tA); one(ax[1],hB,s,tB)
    anim=FuncAnimation(fig,draw,frames=n,interval=1000//fps); anim.save(path,writer=PillowWriter(fps=fps)); plt.close(fig)
gif=f"{DRIVE_ROOT}/gifs/parallel_{EXP_TAG}_{mark}.gif"
save_parallel(nb_run["frames"],b_run["frames"],"NB policy (배터리 무시)","battery policy (절약)",mark,gif)
from IPython.display import Image, display
print("saved ->",gif); display(Image(gif))